In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================
!pip install -q requests openai transformers peft accelerate

In [2]:
# ============================================================
# CELL 2 — MOUNT DRIVE AND CONFIGURATION
# ============================================================
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive')

import os
import shutil

# ---- Secrets ----
GITHUB_TOKEN   = userdata.get('GITHUB_TOKEN')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')   # set in Colab Secrets sidebar

# ---- Paths ----
DRIVE_ROOT  = '/content/drive/MyDrive/298B_WB2'
ADAPTER_DIR = f'{DRIVE_ROOT}/planner_lora_adapter'
EVAL_DIR    = f'{DRIVE_ROOT}/evals/planner'
EVAL_OUT    = f'{EVAL_DIR}/planner_v1_geval.json'
os.makedirs(EVAL_DIR, exist_ok=True)   # create evals/planner folder on Drive

shutil.copy(f'{DRIVE_ROOT}/treesitter_index.json', '/content/treesitter_index.json')
TREESITTER_INDEX_PATH = '/content/treesitter_index.json'

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'

# GitHub — kept for reference but not used in CSV eval path
REPO_OWNER       = 'apache'
REPO_NAME        = 'airflow'

YES_SAMPLES    = 25
NO_SAMPLES     = 25
MAX_NEW_TOKENS = 400
MAX_SEQ_LEN    = 2048
TS_MAX_FILES   = 10
TS_MAX_TOKENS  = 640
SEED = 42
print('Config loaded.')
print(f'Eval output dir: {EVAL_DIR}')


Mounted at /content/drive
Config loaded.
Eval output dir: /content/drive/MyDrive/298B_WB2/evals/planner


In [3]:
# ============================================================
# CELL 3 — IMPORTS
# ============================================================
import json
import re
import time
import random
import requests
import warnings
from pathlib import Path

import torch
import openai
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

warnings.filterwarnings('ignore')
random.seed(SEED)

client = openai.OpenAI(api_key=OPENAI_API_KEY)

assert Path(TREESITTER_INDEX_PATH).exists(), 'Upload treesitter_index.json first'
assert Path(ADAPTER_DIR).exists(), f'Adapter not found at {ADAPTER_DIR}'
print('All paths verified.')

All paths verified.


In [4]:
# ============================================================
# CELL 5 — LOAD EVAL SAMPLES FROM TRAINING SPLIT
#
# planner_pairs.csv has metadata (label, issue_number, output, input_preview)
# input_preview is truncated to 300 chars — NOT suitable for inference
#
# planner_training.jsonl has full text (input+output in chat format)
# We extract the full input from there and pair it with CSV metadata
# ============================================================
import json, random
import pandas as pd
from sklearn.model_selection import train_test_split

# Load CSV for metadata (labels, issue numbers, reference outputs)
pairs_df = pd.read_csv(f'{DRIVE_ROOT}/planner_pairs.csv')

# Load JSONL for full text — extract input portion from chat template
full_inputs = {}   # issue_number -> full input text
with open(f'{DRIVE_ROOT}/planner_training.jsonl') as f:
    jsonl_rows = [json.loads(l) for l in f]

# Match JSONL rows to CSV rows by position (same shuffle seed was used)
# Both have same length and same order since JSONL was written from all_pairs
for idx, (_, row) in enumerate(pairs_df.iterrows()):
    if idx < len(jsonl_rows):
        full_text = jsonl_rows[idx]['text']
        # Extract input portion — between user tags
        match = full_text.split('<|im_end|>')[0]
        input_text = match.replace('<|im_start|>user\n', '').strip()
        full_inputs[int(row['issue_number'])] = input_text

# Reproduce same eval split as training (same seed, same stratify)
_, eval_df = train_test_split(
    pairs_df, test_size=0.10,
    stratify=pairs_df['label'],
    random_state=42
)

yes_eval = eval_df[eval_df['label'] == 'YES'].head(YES_SAMPLES)
no_eval  = eval_df[eval_df['label'] == 'NO'].head(NO_SAMPLES)

# Build sample dicts with full input text
def build_sample(row):
    inum = int(row['issue_number'])
    return {
        'issue_number':   inum,
        'label':          row['label'],
        'full_input':     full_inputs.get(inum, row['input_preview']),  # fallback to preview
        'reference_output': row['output'],   # PR-derived plan from training
    }

yes_samples = [build_sample(r) for _, r in yes_eval.head(25).iterrows()]
no_samples  = [build_sample(r) for _, r in no_eval.head(25).iterrows()]

print(f'YES eval samples: {len(yes_samples)}')
print(f'NO eval samples:  {len(no_samples)}')
print(f'Sample YES issue: #{yes_samples[0]["issue_number"]}')
print(f'Full input length: {len(yes_samples[0]["full_input"])} chars')
print(f'Reference output snippet: {yes_samples[0]["reference_output"][:80]}')


YES eval samples: 25
NO eval samples:  25
Sample YES issue: #62781
Full input length: 4858 chars
Reference output snippet: REQUIRES_CODE_CHANGE: YES | CONFIDENCE: HIGH | REASON: Variable table handle lon


In [5]:
# ============================================================
# CELL 7 — LOAD TREE-SITTER INDEX AND HELPERS
# Exact same functions used during training — ensures consistency
# ============================================================

with open(TREESITTER_INDEX_PATH) as f:
    TS_INDEX = json.load(f)
print(f'Tree-sitter index: {len(TS_INDEX)} files')

NOISY = ['generated.py', 'devel-common/', 'tests_common/', 'pytest_plugin', 'datamodels/', 'airflow-ctl/']

def is_noisy(fp): return any(p in fp.lower() for p in NOISY)

def extract_keywords(text):
    kws = set()
    kws.update(w.lower() for w in re.findall(r'\b[A-Z][a-zA-Z0-9]+\b', text))
    kws.update(re.findall(r'\b[a-z][a-z0-9]*(?:_[a-z0-9]+){1,}\b', text.lower()))
    for kw in ['scheduler','executor','dag','task','operator','sensor','hook',
               'provider','trigger','xcom','variable','connection','webserver','api','model']:
        if kw in text.lower(): kws.add(kw)
    return kws

def get_ts_subset(text):
    kws = extract_keywords(text)
    if not kws: return {}
    scored = []
    for fp, syms in TS_INDEX.items():
        if is_noisy(fp): continue
        score  = sum(2 for kw in kws if kw in fp.lower())
        score += sum(1 for sym in syms.get('classes',[]) + syms.get('functions',[])
                     for kw in kws if kw in sym.lower())
        if score > 0: scored.append((score, fp, syms))
    scored.sort(reverse=True, key=lambda x: x[0])
    return {fp: syms for _, fp, syms in scored[:TS_MAX_FILES]}

def format_ts(subset):
    lines = []
    for fp, syms in subset.items():
        parts = []
        if syms.get('classes'):   parts.append(f"classes: {', '.join(syms['classes'][:5])}")
        if syms.get('functions'): parts.append(f"functions: {', '.join(syms['functions'][:8])}")
        if parts: lines.append(f"{fp} | {' | '.join(parts)}")
    return '\n'.join(lines)

def trunc(text, max_tokens):
    mc = max_tokens * 4
    return text[:mc] + '...' if len(text) > mc else text

print('Tree-sitter helpers loaded.')

Tree-sitter index: 1957 files
Tree-sitter helpers loaded.


In [6]:
# ============================================================
# CELL 8 — LOAD TRAINED MODEL
# Loads base model + planner LoRA adapter from Drive
# ============================================================

print(f'Loading base model: {BASE_MODEL}')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)
print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')
print('Model ready.')

Loading base model: Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading LoRA adapter...
VRAM: 15.5GB
Model ready.


In [7]:
# ============================================================
# CELL 9 — INFERENCE FUNCTION
# run_inference_from_input is the primary function used in eval
# Takes the full pre-built input string (from training JSONL)
# Same format the model was trained on — no reconstruction needed
# ============================================================

def run_inference_from_input(input_text: str) -> str:
    prompt     = f'<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n'
    encoded    = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_ids  = encoded['input_ids']
    attn_mask  = encoded['attention_mask']   # fixes attention mask warning

    with torch.no_grad():
        out = model.generate(
            input_ids,
            attention_mask=attn_mask,        # pass mask explicitly
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,                 # ignored when do_sample=False but silences warning
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)


def extract_predicted_files(model_output: str) -> list:
    """Parse file paths from model output Target files section."""
    files = []
    in_files = False
    for line in model_output.split('\n'):
        if 'target files' in line.lower():
            in_files = True
            continue
        if in_files:
            line = line.strip().lstrip('- ').strip()
            if line and ('/' in line or line.endswith('.py') or line.endswith('.ts')):
                files.append(line)
            elif line and not line.startswith('-') and in_files and len(files) > 0:
                break
    return files


def extract_reference_files(reference_output: str) -> list:
    """Parse file paths from the reference output (CSV output column)."""
    # Reference output uses ' | ' as newline replacement
    text = reference_output.replace(' | ', '\n')
    return extract_predicted_files(text)


print('Inference functions ready.')
# Quick sanity test
test_out = "REQUIRES_CODE_CHANGE: YES\nPLAN:\n- Target files:\n  - airflow/scheduler/scheduler_job_runner.py\n  - airflow/models/dagrun.py"
print('File extraction test:', extract_predicted_files(test_out))


Inference functions ready.
File extraction test: ['airflow/scheduler/scheduler_job_runner.py', 'airflow/models/dagrun.py']


In [8]:
# ============================================================
# CELL 10 — G-EVAL FUNCTION
#
# G-Eval scores on 4 dimensions (1-5 each):
#   ROUTING       — correct YES/NO decision
#   FILE_RELEVANCE — predicted files relevant to the issue
#   PLAN_COHERENCE — root cause and strategy make sense
#   HALLUCINATION  — no invented file paths or facts
#
# YES samples: G-Eval sees model output vs reference (PR body + file list)
# NO samples:  G-Eval sees model output only — checks routing and reasoning
#              (no reference needed since there's no code change ground truth)
# ============================================================

GEVAL_PROMPT_YES = """
You are evaluating a code patch planning model for Apache Airflow GitHub issues.

ISSUE:
{issue_text}

MODEL OUTPUT:
{model_output}

REFERENCE (from actual merged PR — ground truth):
Files actually changed: {reference_files}
PR description: {reference_pr_body}

Score on these four dimensions (1-5 each):

1. ROUTING (1-5): Did the model correctly output REQUIRES_CODE_CHANGE: YES?
   5=correct with high confidence | 3=correct but weak | 1=wrong decision

2. FILE_RELEVANCE (1-5): Are the predicted target files relevant to this issue?
   5=matches reference files closely, all relevant
   3=some relevant files, some misses
   1=files irrelevant or completely wrong module

3. PLAN_COHERENCE (1-5): Does the plan make logical sense for fixing this issue?
   5=root cause accurate, strategy actionable and specific
   3=partially correct or vague
   1=incoherent or wrong

4. HALLUCINATION (1-5): Does the model avoid making up file paths or facts?
   5=all file paths plausible Airflow paths, no invented facts
   3=one or two questionable paths
   1=multiple hallucinated paths

Output exactly this JSON and nothing else:
{{
  "routing": <1-5>,
  "file_relevance": <1-5>,
  "plan_coherence": <1-5>,
  "hallucination": <1-5>,
  "reasoning": "<one sentence on biggest weakness>"
}}
"""

GEVAL_PROMPT_NO = """
You are evaluating a code patch planning model for Apache Airflow GitHub issues.

ISSUE:
{issue_text}

MODEL OUTPUT:
{model_output}

GROUND TRUTH: This issue was definitively closed WITHOUT any code change.
No merged PR exists for this issue. The ground truth label is NO CODE CHANGE NEEDED.
You must accept this as fact regardless of how the issue text reads.
Your job is NOT to decide if the issue needs a code change — that has already been determined.
Your job is only to evaluate how well the model responded given that NO is the correct answer.

Score on these dimensions (1-5 each):

1. ROUTING (1-5): Did the model output REQUIRES_CODE_CHANGE: NO?
   5 = correctly output NO
   1 = incorrectly output YES (wrong routing)
   Do NOT penalise the model for saying NO — that IS the correct answer.

2. FILE_RELEVANCE (1-5):
   5 = model said NO and produced no spurious file list
   1 = model said YES and produced an irrelevant file list
   If model correctly said NO, default score is 5.

3. PLAN_COHERENCE (1-5): Quality of the REASON given for NO.
   5 = specific reason referencing issue content (e.g. "this is a docs request", "config change only")
   3 = generic reason ("no code change needed")
   1 = no reason given, or model said YES instead

4. HALLUCINATION (1-5):
   5 = model cleanly stopped at NO with reason, no spurious plan generated
   1 = model generated a full code plan despite NO being correct

Output exactly this JSON and nothing else:
{{
  "routing": <1-5>,
  "file_relevance": <1-5>,
  "plan_coherence": <1-5>,
  "hallucination": <1-5>,
  "reasoning": "<one sentence on biggest weakness>"
}}
"""


def geval_score(issue_text: str, model_output: str,
                reference_files: list = None,
                reference_pr_body: str = None,
                label: str = 'YES') -> dict:
    if label == 'YES':
        prompt = GEVAL_PROMPT_YES.format(
            issue_text=issue_text[:1200],
            model_output=model_output,
            reference_files=', '.join(reference_files or []),
            reference_pr_body=(reference_pr_body or '')[:500],
        )
    else:
        prompt = GEVAL_PROMPT_NO.format(
            issue_text=issue_text[:1200],
            model_output=model_output,
        )

    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    try:
        return json.loads(resp.choices[0].message.content)
    except Exception:
        raw = resp.choices[0].message.content
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {}


print('G-Eval function ready.')

G-Eval function ready.


In [9]:
# ============================================================
# CELL 11 — RUN EVALUATION ON YES SAMPLES
# Uses full_input (from JSONL) for inference
# Uses reference_output (from CSV) as G-Eval ground truth
# Extracts reference files from reference_output for FILE_RELEVANCE scoring
# Robust: retries on RateLimitError with exponential backoff
#         saves partial results so a crash doesn't lose progress
# ============================================================
import time

yes_results = []

print('=' * 60)
print('EVALUATING YES SAMPLES')
print('=' * 60)

def geval_score_with_retry(issue_text, model_output, reference_files=None,
                            reference_pr_body=None, label='YES', max_retries=5):
    """Wrap geval_score with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return geval_score(
                issue_text=issue_text,
                model_output=model_output,
                reference_files=reference_files,
                reference_pr_body=reference_pr_body,
                label=label
            )
        except Exception as e:
            if 'rate_limit' in str(e).lower() or '429' in str(e):
                wait = (2 ** attempt) * 5 + random.uniform(0, 2)
                print(f'  Rate limit hit — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                print(f'  G-Eval error: {e}')
                return {}
    print('  Max retries exceeded — skipping G-Eval for this sample')
    return {}

for i, sample in enumerate(yes_samples):
    # Inference on full input
    model_output = run_inference_from_input(sample['full_input'])

    # Issue text for G-Eval = first part of full input (before REPO_SYMBOLS)
    issue_text = sample['full_input'].split('REPO_SYMBOLS:')[0].strip()

    # Extract reference files from the PR-derived reference output
    ref_files = extract_reference_files(sample['reference_output'])
    ref_body  = sample['reference_output'][:500]

    # G-Eval with retry on rate limit
    scores = geval_score_with_retry(
        issue_text=issue_text,
        model_output=model_output,
        reference_files=ref_files,
        reference_pr_body=ref_body,
        label='YES'
    )

    # Small delay between calls to avoid hitting TPM limit
    time.sleep(2)

    # Routing binary: did model say YES
    said_yes = 'REQUIRES_CODE_CHANGE: YES' in model_output

    result = {
        'issue_number':    sample['issue_number'],
        'label':           'YES',
        'predicted_yes':   said_yes,
        'model_output':    model_output,
        'reference_files': ref_files,
        **scores
    }
    yes_results.append(result)

    # Save partial results after every sample — crash-safe
    with open(f'{EVAL_DIR}/yes_results_partial.json', 'w') as f:
        json.dump(yes_results, f, indent=2)

    avg = {d: sum(r.get(d, 0) for r in yes_results) / len(yes_results)
           for d in ['routing', 'file_relevance', 'plan_coherence', 'hallucination']}

    print(
        f'[{i+1:2}/{len(yes_samples)}] #{sample["issue_number"]}: '
        f'Routing={scores.get("routing","?")}/FileRelevance={scores.get("file_relevance","?")}'
        f'/PlanCoherence={scores.get("plan_coherence","?")}/Hallucination={scores.get("hallucination","?")} '
        f'said_YES={said_yes} | '
        f'avg Routing={avg["routing"]:.2f} FileRelevance={avg["file_relevance"]:.2f} '
        f'PlanCoherence={avg["plan_coherence"]:.2f}'
    )

print('\n=== YES SAMPLES FINAL AVERAGES ===')
for d in ['routing', 'file_relevance', 'plan_coherence', 'hallucination']:
    vals = [r.get(d, 0) for r in yes_results]
    print(f'  {d}: avg={sum(vals)/len(vals):.2f} min={min(vals)} max={max(vals)}')

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


EVALUATING YES SAMPLES
[ 1/25] #62781: Routing=5/FileRelevance=3/PlanCoherence=4/Hallucination=4 said_YES=True | avg Routing=5.00 FileRelevance=3.00 PlanCoherence=4.00
[ 2/25] #57717: Routing=5/FileRelevance=1/PlanCoherence=1/Hallucination=1 said_YES=True | avg Routing=5.00 FileRelevance=2.00 PlanCoherence=2.50
[ 3/25] #60049: Routing=5/FileRelevance=3/PlanCoherence=4/Hallucination=3 said_YES=True | avg Routing=5.00 FileRelevance=2.33 PlanCoherence=3.00
[ 4/25] #56893: Routing=5/FileRelevance=3/PlanCoherence=4/Hallucination=3 said_YES=True | avg Routing=5.00 FileRelevance=2.50 PlanCoherence=3.25
[ 5/25] #60751: Routing=1/FileRelevance=1/PlanCoherence=1/Hallucination=1 said_YES=False | avg Routing=4.20 FileRelevance=2.20 PlanCoherence=2.80
[ 6/25] #62345: Routing=5/FileRelevance=5/PlanCoherence=3/Hallucination=5 said_YES=True | avg Routing=4.33 FileRelevance=2.67 PlanCoherence=2.83
[ 7/25] #59028: Routing=5/FileRelevance=3/PlanCoherence=3/Hallucination=3 said_YES=True | avg Routing=4.43

In [10]:
# ============================================================
# CELL 12 — RUN EVALUATION ON NO SAMPLES
# NO samples have no reference files — just routing and reasoning
# Robust: retries on RateLimitError, saves partial results
# ============================================================
import time

no_results = []

print('=' * 60)
print('EVALUATING NO SAMPLES')
print('=' * 60)

for i, sample in enumerate(no_samples):
    # Inference on full input
    model_output = run_inference_from_input(sample['full_input'])

    # Issue text for G-Eval
    issue_text = sample['full_input'].split('REPO_SYMBOLS:')[0].strip()

    # G-Eval with retry — no reference files needed for NO samples
    scores = geval_score_with_retry(
        issue_text=issue_text,
        model_output=model_output,
        label='NO'
    )

    # Small delay between calls to avoid TPM limit
    time.sleep(2)

    said_no = 'REQUIRES_CODE_CHANGE: NO' in model_output

    result = {
        'issue_number':  sample['issue_number'],
        'label':         'NO',
        'predicted_yes': not said_no,   # False = correctly said NO
        'model_output':  model_output,
        **scores
    }
    no_results.append(result)

    # Save partial results after every sample — crash-safe
    with open(f'{EVAL_DIR}/no_results_partial.json', 'w') as f:
        json.dump(no_results, f, indent=2)

    print(
        f'[{i+1:2}/{len(no_samples)}] #{sample["issue_number"]}: '
        f'Routing={scores.get("routing","?")}/FileRelevance={scores.get("file_relevance","?")}'
        f'/PlanCoherence={scores.get("plan_coherence","?")}/Hallucination={scores.get("hallucination","?")} '
        f'said_NO={said_no} | {scores.get("reasoning","")[:70]}'
    )

print('\n=== NO SAMPLES FINAL AVERAGES ===')
for d in ['routing', 'file_relevance', 'plan_coherence', 'hallucination']:
    vals = [r.get(d, 0) for r in no_results]
    print(f'  {d}: avg={sum(vals)/len(vals):.2f} min={min(vals)} max={max(vals)}')

EVALUATING NO SAMPLES
[ 1/25] #61825: Routing=5/FileRelevance=5/PlanCoherence=3/Hallucination=5 said_NO=True | The model provided a generic reason for no code change, lacking specif
[ 2/25] #58309: Routing=5/FileRelevance=5/PlanCoherence=3/Hallucination=5 said_NO=True | The reason given was generic and did not reference specific issue cont
[ 3/25] #58856: Routing=1/FileRelevance=1/PlanCoherence=1/Hallucination=1 said_NO=False | The model incorrectly determined that a code change was needed and gen
[ 4/25] #63218: Routing=1/FileRelevance=1/PlanCoherence=1/Hallucination=1 said_NO=False | The model incorrectly determined that a code change was needed and gen
[ 5/25] #59958: Routing=5/FileRelevance=5/PlanCoherence=3/Hallucination=5 said_NO=True | The model provided a generic reason for NO, lacking specific reference
[ 6/25] #63726: Routing=1/FileRelevance=1/PlanCoherence=1/Hallucination=1 said_NO=False | The model incorrectly determined that a code change was needed and pro
[ 7/25] #58541:

In [11]:
# ============================================================
# CELL 13 — COMBINED SUMMARY, F1, PLOTS AND SAVE
#
# F1 for binary routing (REQUIRES_CODE_CHANGE YES/NO):
#   True label:  YES samples = positive class, NO samples = negative class
#   Predicted:   model_output contains 'REQUIRES_CODE_CHANGE: YES' or not
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

all_results = yes_results + no_results

# ── F1 CALCULATION ──
# Ground truth: YES=1, NO=0
# Prediction: did model say REQUIRES_CODE_CHANGE: YES
y_true = [1] * len(yes_results) + [0] * len(no_results)
y_pred = [1 if r['predicted_yes'] else 0 for r in all_results]

TP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
FP = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
FN = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
TN = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy  = (TP + TN) / len(y_true)

print('=' * 60)
print('ROUTING BINARY CLASSIFICATION (YES/NO)')
print('=' * 60)
print(f'  TP={TP} FP={FP} FN={FN} TN={TN}')
print(f'  Precision: {precision:.3f}')
print(f'  Recall:    {recall:.3f}')
print(f'  F1 Score:  {f1:.3f}')
print(f'  Accuracy:  {accuracy:.3f}')

print()
print('=' * 60)
print('G-EVAL DIMENSION AVERAGES (all 50 samples)')
print('=' * 60)
dims = ['routing', 'file_relevance', 'plan_coherence', 'hallucination']
dim_avgs = {}
for d in dims:
    vals = [r.get(d, 0) for r in all_results]
    dim_avgs[d] = sum(vals) / len(vals)
    print(f'  {d:20}: avg={dim_avgs[d]:.2f} | min={min(vals)} max={max(vals)}')

# ── PLOT 1: G-Eval dimension bar chart ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Planner v1 — G-Eval Evaluation Results', fontsize=14, fontweight='bold')

# Bar chart — dimension averages
ax1 = axes[0]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars = ax1.bar(
    [d.replace('_', '\n') for d in dims],
    [dim_avgs[d] for d in dims],
    color=colors, width=0.5, edgecolor='white'
)
ax1.set_ylim(0, 5)
ax1.set_ylabel('Score (1-5)')
ax1.set_title('G-Eval Dimension Averages')
ax1.axhline(y=3.5, color='green', linestyle='--', alpha=0.5, label='Good threshold')
ax1.axhline(y=4.0, color='blue', linestyle='--', alpha=0.5, label='Strong threshold')
for bar, val in zip(bars, [dim_avgs[d] for d in dims]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
ax1.legend(fontsize=8)

# ── PLOT 2: F1 confusion matrix heatmap ──
ax2 = axes[1]
cm = np.array([[TN, FP], [FN, TP]])
im = ax2.imshow(cm, cmap='Blues')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(['Pred NO', 'Pred YES'])
ax2.set_yticklabels(['True NO', 'True YES'])
ax2.set_title(f'Routing Confusion Matrix\nF1={f1:.3f} | Acc={accuracy:.3f}')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i, j]), ha='center', va='center',
                 fontsize=16, fontweight='bold',
                 color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax2)

# ── PLOT 3: Score distribution per dimension (YES vs NO) ──
ax3 = axes[2]
yes_avgs = {d: sum(r.get(d,0) for r in yes_results)/len(yes_results) for d in dims}
no_avgs  = {d: sum(r.get(d,0) for r in no_results)/len(no_results)  for d in dims}

x = np.arange(len(dims))
width = 0.35
b1 = ax3.bar(x - width/2, [yes_avgs[d] for d in dims], width, label='YES samples', color='#2196F3', alpha=0.8)
b2 = ax3.bar(x + width/2, [no_avgs[d]  for d in dims], width, label='NO samples',  color='#F44336', alpha=0.8)
ax3.set_ylim(0, 5)
ax3.set_ylabel('Score (1-5)')
ax3.set_title('Scores: YES vs NO Samples')
ax3.set_xticks(x)
ax3.set_xticklabels([d.replace('_','\n') for d in dims])
ax3.legend()
ax3.axhline(y=3.5, color='green', linestyle='--', alpha=0.4)

plt.tight_layout()
plot_path = f'{EVAL_DIR}/geval_results.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

# ── PLOT 4: Per-issue score heatmap for YES samples ──
fig2, ax4 = plt.subplots(figsize=(14, max(6, len(yes_results) * 0.3)))
score_matrix = np.array([[r.get(d, 0) for d in dims] for r in yes_results])
im2 = ax4.imshow(score_matrix, cmap='RdYlGn', vmin=1, vmax=5, aspect='auto')
ax4.set_xticks(range(len(dims)))
ax4.set_xticklabels([d.replace('_','\n') for d in dims])
ax4.set_yticks(range(len(yes_results)))
ax4.set_yticklabels([f'#{r["issue_number"]}' for r in yes_results], fontsize=7)
ax4.set_title('Per-Issue G-Eval Scores — YES Samples')
plt.colorbar(im2, ax=ax4, label='Score (1-5)')
for i in range(len(yes_results)):
    for j in range(len(dims)):
        ax4.text(j, i, str(int(score_matrix[i, j])), ha='center', va='center', fontsize=7)
plt.tight_layout()
heatmap_path = f'{EVAL_DIR}/per_issue_scores.png'
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Heatmap saved: {heatmap_path}')

# ── SAVE JSON RESULTS ──
with open(EVAL_OUT, 'w') as f:
    json.dump({
        'yes_results':       yes_results,
        'no_results':        no_results,
        'adapter_dir':       ADAPTER_DIR,
        'f1':                round(f1, 4),
        'precision':         round(precision, 4),
        'recall':            round(recall, 4),
        'accuracy':          round(accuracy, 4),
        'confusion_matrix':  {'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN},
        'dim_averages':      {d: round(dim_avgs[d], 4) for d in dims},
    }, f, indent=2)
print(f'Results saved: {EVAL_OUT}')
print(f'\nAll eval outputs in: {EVAL_DIR}')


ROUTING BINARY CLASSIFICATION (YES/NO)
  TP=22 FP=11 FN=3 TN=14
  Precision: 0.667
  Recall:    0.880
  F1 Score:  0.759
  Accuracy:  0.720

G-EVAL DIMENSION AVERAGES (all 50 samples)
  routing             : avg=3.88 | min=1 max=5
  file_relevance      : avg=3.16 | min=1 max=5
  plan_coherence      : avg=2.62 | min=1 max=5
  hallucination       : avg=3.52 | min=1 max=5
Plot saved: /content/drive/MyDrive/298B_WB2/evals/planner/geval_results.png
Heatmap saved: /content/drive/MyDrive/298B_WB2/evals/planner/per_issue_scores.png
Results saved: /content/drive/MyDrive/298B_WB2/evals/planner/planner_v1_geval.json

All eval outputs in: /content/drive/MyDrive/298B_WB2/evals/planner


In [12]:
# ============================================================
# CELL 14 — PRINT WEAKEST EXAMPLES FOR MANUAL REVIEW
# Shows the 5 YES examples with lowest file_relevance scores
# These are the most informative for diagnosing model weaknesses
# ============================================================

print('TOP 5 WEAKEST YES EXAMPLES (by file_relevance):')
print('=' * 60)
weakest = sorted(yes_results, key=lambda r: r.get('file_relevance', 5))[:5]
for r in weakest:
    print(f"Issue #{r['issue_number']}: file_relevance={r.get('file_relevance')} ")
    print(f"  Predicted files: {extract_predicted_files(r['model_output'])}")
    print(f"  Reference files: {r['reference_files'][:3]}")
    print(f"  Reasoning: {r.get('reasoning','')}")
    print()


TOP 5 WEAKEST YES EXAMPLES (by file_relevance):
Issue #57717: file_relevance=1 
  Predicted files: ['airflow-core/src/airflow/ui/tests/e2e/pages/AdminPage.ts', 'airflow-core/src/airflow/ui/src/layouts/Nav/NavLinks.tsx', 'airflow-core/src/airflow/ui/src/layouts']
  Reference files: ['airflow-core/src/airflow/api_fastapi/core_api/services/ui/connections.py']
  Reasoning: The model's biggest weakness is its incoherent plan and incorrect file paths, which do not match the reference or make logical sense for the issue.

Issue #60751: file_relevance=1 
  Predicted files: []
  Reference files: ['chart/files/pod-template-file.kubernetes-helm-yaml', 'chart/templates/workers/worker-deployment.yaml', 'helm-tests/tests/helm_tests/airflow_aux/test_pod_template_file.py']
  Reasoning: The model incorrectly determined that no code change was required, and it failed to identify relevant files or provide a coherent plan for addressing the issue.

Issue #62909: file_relevance=1 
  Predicted files: ['task